# CEG-WM Content V4 — whitened-LF full-runtime canary handoff

This Colab notebook runs the frozen one-unit, non-roster operational canary from execution source `stage-a-content-adaptive-dual-branch-v4-whitened-lf-canary@4cecf321a20b0e50baf6d47057d99aec57802b55`. It mounts Drive first, checks out and verifies that exact source, installs only the checked-out project, reads `CEG_WM_ROOT_KEY` and `HF_TOKEN` from Colab Secrets, and invokes the existing canary runner exactly once.

The handoff is engineering-only: `formal_roster_member=false`, `scientific_denominator_units=0`, and claim ceiling `full_non_roster_runtime_canary_only`. On success it creates one result JSON and SHA-256 sidecar in the frozen Drive destination. Run the cells once from top to bottom and stop after any failure or interruption.


## 1. Mount Drive, then prove the fresh execution checkout

Drive mounting is the first external action. The source path and exact-bound result destination must both be fresh before checkout or installation.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except BaseException:
    print('CEGWM_CONTENT_V4_CANARY_HANDOFF_FAILURE {"canary_id":"content-v4-whitened-lf-full-runtime-non-roster-canary-v1","error_class":"OtherOperationalError","execution_exact":"4cecf321a20b0e50baf6d47057d99aec57802b55","stage":"drive_mount","status":"operational_failure"}', flush=True)
    HANDOFF_FAILED = True
else:
    HANDOFF_FAILED = False

import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-adaptive-dual-branch-v4-whitened-lf-canary"
EXACT = "4cecf321a20b0e50baf6d47057d99aec57802b55"
RUNNER_MODULE = "experiments.run_content_v4_whitened_lf_canary"
RUNNER_PREFIX = "CEGWM_CONTENT_V4_CANARY_RESULT"
FAILURE_PREFIX = "CEGWM_CONTENT_V4_CANARY_HANDOFF_FAILURE"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V4_CANARY_ARTIFACT"
CANARY_ID = "content-v4-whitened-lf-full-runtime-non-roster-canary-v1"
CLAIM_CEILING = "full_non_roster_runtime_canary_only"
PROTOCOL_ID = "cegwm-stage-a-content-v4-whitened-lf-adaptive-hf-clean-v1"
PROTOCOL_DIGEST = "a9fdf3e5d384976c11bbd542c3248483806473ed1dca91dc5d753ab10ec5beb0"
METHOD_ID = "content_v4_clean_null_whitened_lf_adaptive_hf_v1"
CANDIDATE_ID = "content_v4_clean_null_whitened_lf_adaptive_hf_semantic_gate_v1"
UNIT_ID = "content-v4-whitened-lf-canary-0001"
SOURCE_ID = "content-v4-whitened-lf-canary-prompt-9001"
SEED = 1415149
HEIGHT = 512
WIDTH = 512
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
DINO_ASSET_ID = "facebook/dinov2-small"
LF_SCORER_ID = "content_v4_whitened_lf_dct_matched_cosine_v1"
OBSERVATION_CONTRACT_ID = "final_rgb_current_processor_sd35_vae_posterior_mode_float32_1x16x64x64_v1"
ASSET_ROLE_ID = "content_v4_clean_null_whitening_operator_v1"
ASSET_SCHEMA_ID = "cegwm_content_v4_clean_null_whitening_operator_asset_v1"
ASSET_SHA256 = "a7021dd8b98bc4282b98ed5d1fe276236d99a3c9e80b9bdce015d28cf715633f"
ASSET_SIDECAR_SHA256 = "c900cce0980348eeadcf07d782b6169c4d46ac55d7154db0fc0a0a878cce0ced"
HANDOFF_EXECUTION_SOURCE = BRANCH + "@" + EXACT

repo = pathlib.Path("/content/cegwm-content-v4-whitened-lf-canary-source")
drive_root = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v4_whitened_lf_canary")
result_dir = drive_root / CANARY_ID / EXACT
result_path = result_dir / (CANARY_ID + ".json")
checksum_path = result_dir / (CANARY_ID + ".json.sha256")
RUNNER_ATTEMPTED = False

_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "UnicodeDecodeError", "ValueError",
}

def fail(stage, error_class="RuntimeError"):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    HANDOFF_FAILED = True
    payload = {
        "status": "operational_failure",
        "canary_id": CANARY_ID,
        "execution_exact": EXACT,
        "stage": stage,
        "error_class": error_class,
    }
    print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists() or result_dir.exists() or result_path.exists() or checksum_path.exists():
            raise FileExistsError
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
        ):
            raise RuntimeError
    except BaseException as error:
        fail("source_checkout_identity_validation", type(error).__name__)


## 2. Install only the checked-out project

The named branch, execution exact, clean state, and absent result destination are rechecked after installation.


In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or result_dir.exists()
            or result_path.exists()
            or checksum_path.exists()
        ):
            raise RuntimeError
    except BaseException as error:
        fail("dependency_install_and_source_recheck", type(error).__name__)


## 3. Invoke once, validate the bounded result, and persist the create-only pair

Secrets enter only the child environment. Raw child streams are never printed. A validated successful result is wrapped with the frozen handoff identity and persisted natively to Drive; any failure leaves no result pair.


In [ ]:
import math
import os
import re
from google.colab import userdata

RUNNER_SUCCESS_FIELDS = {
    "status", "claim_ceiling", "canary_id", "unit_id", "source_id",
    "exact", "public_key_digest", "run_id", "protocol_id",
    "protocol_digest", "content_method_id", "evaluated_candidate_id",
    "model_id", "dino_asset_id", "lf_scorer_id",
    "observation_contract_id", "whitening_asset_role_id",
    "whitening_asset_schema_id", "whitening_asset_sha256",
    "whitening_asset_sidecar_sha256", "seed", "height", "width",
    "combined_actual_dtype_relative_l2", "lf_effective_relative_l2",
    "hf_effective_relative_l2", "lf_branch_share", "hf_branch_share",
    "minimum_counterfactual_effect", "probe_evaluation_count",
    "paired_rgb_psnr_db", "joint_registered_lf_score",
    "joint_registered_hf_score", "joint_registered_joint_score",
    "primary_null_registered_lf_score", "primary_null_registered_hf_score",
    "primary_null_registered_joint_score", "formal_roster_member",
    "scientific_denominator_units",
}

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    CAPTURE_LIMIT = 4096
    try:
        if result_dir.exists() or result_path.exists() or checksum_path.exists():
            raise FileExistsError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
        ):
            raise RuntimeError
        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip():
            raise RuntimeError
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
            ],
            cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None

    accepted_result = None
    if launch_error is not None:
        captured.clear()
        fail("canary_runner_launch", launch_error)
    elif runner_rc != 0:
        captured.clear()
        fail("canary_runner_nonzero")
    elif capture_overflow:
        captured.clear()
        fail("canary_runner_stdout_overflow")
    else:
        try:
            captured_text = captured.decode("utf-8", errors="strict")
            if captured_text.count("\n") != 1 or not captured_text.endswith("\n"):
                raise RuntimeError
            runner_line = captured_text[:-1]
            if len(runner_line) > CAPTURE_LIMIT or not runner_line.startswith(RUNNER_PREFIX + " "):
                raise RuntimeError
            payload = json.loads(runner_line.split(" ", 1)[1])
            if not isinstance(payload, dict) or set(payload) != RUNNER_SUCCESS_FIELDS:
                raise RuntimeError
            numeric_fields = (
                "combined_actual_dtype_relative_l2", "lf_effective_relative_l2",
                "hf_effective_relative_l2", "lf_branch_share", "hf_branch_share",
                "minimum_counterfactual_effect", "paired_rgb_psnr_db",
                "joint_registered_lf_score", "joint_registered_hf_score",
                "joint_registered_joint_score", "primary_null_registered_lf_score",
                "primary_null_registered_hf_score", "primary_null_registered_joint_score",
            )
            integer_fields = (
                "seed", "height", "width", "probe_evaluation_count",
                "scientific_denominator_units",
            )
            if (
                payload["status"] != "operational_canary_pass"
                or payload["canary_id"] != CANARY_ID
                or payload["unit_id"] != UNIT_ID
                or payload["source_id"] != SOURCE_ID
                or payload["exact"] != EXACT
                or payload["claim_ceiling"] != CLAIM_CEILING
                or payload["protocol_id"] != PROTOCOL_ID
                or payload["protocol_digest"] != PROTOCOL_DIGEST
                or payload["content_method_id"] != METHOD_ID
                or payload["evaluated_candidate_id"] != CANDIDATE_ID
                or payload["model_id"] != MODEL_ID
                or payload["dino_asset_id"] != DINO_ASSET_ID
                or payload["lf_scorer_id"] != LF_SCORER_ID
                or payload["observation_contract_id"] != OBSERVATION_CONTRACT_ID
                or payload["whitening_asset_role_id"] != ASSET_ROLE_ID
                or payload["whitening_asset_schema_id"] != ASSET_SCHEMA_ID
                or payload["whitening_asset_sha256"] != ASSET_SHA256
                or payload["whitening_asset_sidecar_sha256"] != ASSET_SIDECAR_SHA256
                or payload["seed"] != SEED
                or payload["height"] != HEIGHT
                or payload["width"] != WIDTH
                or payload["formal_roster_member"] is not False
                or payload["scientific_denominator_units"] != 0
                or payload["probe_evaluation_count"] != 64
                or re.fullmatch(r"[0-9a-f]{64}", payload["public_key_digest"]) is None
                or re.fullmatch(r"content-v4-a9fdf3e5d384-[0-9a-f]{12}", payload["run_id"]) is None
                or any(isinstance(payload[name], bool) for name in numeric_fields + integer_fields)
                or any(not isinstance(payload[name], (int, float)) for name in numeric_fields)
                or any(not isinstance(payload[name], int) for name in integer_fields)
                or any(not math.isfinite(float(payload[name])) for name in numeric_fields)
            ):
                raise RuntimeError
            accepted_result = payload
        except BaseException as error:
            fail("canary_runner_result_validation", type(error).__name__)
        finally:
            captured.clear()


## 4. Persist the validated result pair

This runner-free cell writes the exact-bound JSON and SHA-256 sidecar with create-only operations, then prints one bounded artifact receipt. It remains silent after any earlier failure.


In [ ]:
import hashlib

if not HANDOFF_FAILED and globals().get("accepted_result") is not None:
    created = []
    made_result_dir = False
    try:
        if result_dir.exists() or result_path.exists() or checksum_path.exists():
            raise FileExistsError
        artifact = {
            "schema_version": "cegwm_content_v4_whitened_lf_canary_handoff_v1",
            "canary_id": CANARY_ID,
            "execution_branch": BRANCH,
            "execution_exact": EXACT,
            "handoff_execution_source": HANDOFF_EXECUTION_SOURCE,
            "formal_roster_member": False,
            "scientific_denominator_units": 0,
            "claim_ceiling": CLAIM_CEILING,
            "bounded_result": accepted_result,
        }
        artifact_bytes = (json.dumps(artifact, sort_keys=True, separators=(",", ":"), allow_nan=False) + "\n").encode("utf-8")
        artifact_sha256 = hashlib.sha256(artifact_bytes).hexdigest()
        checksum_bytes = (artifact_sha256 + "  " + result_path.name + "\n").encode("ascii")
        result_dir.mkdir(parents=True, exist_ok=False)
        made_result_dir = True
        with result_path.open("xb") as handle:
            created.append(result_path)
            handle.write(artifact_bytes)
        with checksum_path.open("xb") as handle:
            created.append(checksum_path)
            handle.write(checksum_bytes)
        receipt = {
            "status": "artifact_persisted",
            "canary_id": CANARY_ID,
            "execution_exact": EXACT,
            "result_path": str(result_path),
            "checksum_path": str(checksum_path),
            "result_sha256": artifact_sha256,
        }
        receipt_line = ARTIFACT_PREFIX + " " + json.dumps(receipt, sort_keys=True, separators=(",", ":"))
        if len(receipt_line) > 4096:
            raise RuntimeError
        print(receipt_line, flush=True)
    except BaseException as error:
        for path in reversed(created):
            path.unlink(missing_ok=True)
        if made_result_dir:
            try:
                result_dir.rmdir()
            except OSError:
                pass
        fail("create_only_artifact_persistence", type(error).__name__)


## Stop boundary

After the single artifact receipt or sanitized failure line, stop. The Notebook does not download, relaunch, or expose raw runner output, tracebacks, Secrets, prompts, or private runtime state. No formal scientific, calibration, fixed-FPR, or promotion conclusion follows from this canary.
